In [1]:
"""
Co-locate ICS records with their associated fire perimeters
"""

# imports
import os, sys
from shapely.geometry import Point

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
boxdir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/'
datadir = boxdir + 'data/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

proj_crs = 'EPSG:5070'

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation


In [2]:
# --- Read in the incidents data
fp = os.path.join(boxdir,'projects/ics209plus/data/spatial/mod/ics209plus-spatial_1999to2023.gpkg') # update, spatial
incis = gpd.read_file(fp)
incis = incis[incis['START_YEAR'] >= 2014] # temporal filter
print(f"There are [{len(incis)}] incidents in the table.")

# Filter incidents with no acres
print(incis['FINAL_ACRES'].quantile(0.10))
incis = incis[incis['FINAL_ACRES'] > 2] # 2 acre threshold?

# Tidy some of the columns
# incis.drop(columns=['Unnamed: 0'], inplace=True)
incis['DISCOVERY_DATE'] = pd.to_datetime(incis['DISCOVERY_DATE'], format="mixed")
incis['FINAL_REPORT_DATE'] = pd.to_datetime(incis['FINAL_REPORT_DATE'], format="mixed")
incis['START_YEAR'] = incis['START_YEAR'].astype(int)

print(f"Columns:\n{incis.columns}")
pd.set_option("display.float_format", "{:,.4f}".format)
print(incis['FINAL_ACRES'].describe())

There are [15969] incidents in the table.
2.0
Columns:
Index(['CAUSE', 'COMPLEX', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
       'EVACUATION_REPORTED', 'EXPECTED_CONTAINMENT_DATE', 'FATALITIES',
       'FATALITIES_PUBLIC', 'FATALITIES_RESPONDER', 'FINAL_ACRES',
       'FINAL_REPORT_DATE', 'FIRE_SIZE_CLASS', 'FIRE_YEAR', 'FOD_CONT_DATE',
       'FOD_CONT_DOY', 'FOD_CONT_TIME', 'FOD_DISCOVERY_DATE',
       'FOD_DISCOVERY_DOY', 'FOD_DISCOVERY_TIME', 'FOD_EXCL_CAT',
       'FOD_FIRE_SIZE', 'FOD_ID', 'FOD_INCIDENT_GROUP_NAME', 'FOD_LATITUDE',
       'FOD_LONGITUDE', 'FUEL_MODEL', 'INCIDENT_DESCRIPTION', 'INCIDENT_ID',
       'INCIDENT_ID_OLD', 'INCIDENT_NAME', 'INCIDENT_NUMBER',
       'INCTYP_ABBREVIATION', 'INC_IDENTIFIER', 'INC_MGMT_NUM_SITREPS',
       'INJURIES_TOTAL', 'IRWIN_ID', 'LL_CONFIDENCE', 'LL_UPDATE',
       'LOCAL_TIMEZONE', 'MTBS_FIRE_NAME', 'MTBS_ID',
       'NWCG_CAUSE_CLASSIFICATION', 'NWCG_GENERAL_CAUSE', 'PEAK_EVACUATIONS',
       'POO_CITY', 'POO_COUNTY', 'POO_LATITUDE', 'P

In [3]:
# --- Check on the ICS Complex records ...
complexes = incis[incis['COMPLEX']=='True']
print(f"Number of complexes: {len(complexes)}")
print(complexes.sort_values(by='FINAL_ACRES', ascending=False)[['INCIDENT_NAME','FINAL_ACRES']].head())

Number of complexes: 195
             INCIDENT_NAME    FINAL_ACRES
34231       AUGUST COMPLEX 1,032,648.0000
36960         LIME COMPLEX   865,625.0000
28874  NW OKLAHOMA COMPLEX   779,292.0000
32487  CHALKYITSIK COMPLEX   505,273.0000
31854    MENDOCINO COMPLEX   459,123.0000


In [4]:
# --- Load the latest incidents table
fp = os.path.join(boxdir, 'projects/ics209plus/data/tabular/ics209-plus-wf_sitreps_1999to2024-wx-draft.csv')

### Join ICS incidents to their MTBS perimeters

In [5]:
# --- Isolate non-complex incidents with an MTBS IDs
incis_mtbs = incis.loc[incis["MTBS_ID"].notna()].copy()
print(f"There are {len(incis_mtbs)} incidents with MTBS IDs.")

# --- Check for duplicate MTBS IDs
dup_counts = incis_mtbs["MTBS_ID"].value_counts()
dup_mtbs = dup_counts[dup_counts > 1]
print(f"\tDuplicated MTBS IDs: {len(dup_mtbs)}")
print(f"\tICS incidents involved: {dup_mtbs.sum()}")

There are 4048 incidents with MTBS IDs.
	Duplicated MTBS IDs: 43
	ICS incidents involved: 100


In [6]:
incis_mtbs['START_YEAR'].max()

np.int64(2022)

In [24]:
# --- Load the MTBS perimeter data
fp = os.path.join(datadir, "wildfire/MTBS/mtbs_perimeter_data/mtbs_perims_DD.shp") # stored locally
mtbs = gpd.read_file(fp).to_crs(incis.crs)
mtbs['Ig_Date'] = pd.to_datetime(mtbs['Ig_Date'])
mtbs['Ig_Year'] = mtbs['Ig_Date'].dt.year
mtbs = mtbs[mtbs['Ig_Year'] >= 2014] # filter to match the incidents table
print(f"MTBS up to {mtbs['Ig_Year'].max()}")
mtbs = mtbs[['Event_ID','Ig_Date','Ig_Year','BurnBndAc','geometry']] # keep relevant columns

# --- Join incidents to MTBS perimeters,
# keeping polygon geometry and GeoDataFrame
incis_mtbs = gpd.GeoDataFrame(
    incis.drop(columns='geometry')
    .merge(mtbs, right_on="Event_ID", left_on='MTBS_ID', how='inner'),
    geometry='geometry',
    crs=incis.crs
)
print(f"Matched [{len(incis_mtbs)}/{len(incis)}]")

# --- Handle duplicated records using burned area difference
incis_mtbs['FINAL_ACRES'] = pd.to_numeric(incis_mtbs['FINAL_ACRES'], errors="coerce")
incis_mtbs["BurnBndAc"] = pd.to_numeric(incis_mtbs["BurnBndAc"], errors="coerce")
incis_mtbs["acre_diff_abs"] = (incis_mtbs['FINAL_ACRES'] - incis_mtbs["BurnBndAc"]).abs()
incis_mtbs["date_diff_days"] = (incis_mtbs["DISCOVERY_DATE"] - incis_mtbs["Ig_Date"]).abs().dt.days
# symmetric percent diff (mean denominator) to avoid oddities
den = (incis_mtbs["FINAL_ACRES"].abs() + incis_mtbs["BurnBndAc"].abs()) / 2.0
incis_mtbs["acre_diff_pct"] = 100.0 * incis_mtbs["acre_diff_abs"] / den.replace(0, np.nan)

# pick "best" row per MTBS_ID
sort_cols = ['MTBS_ID', 'date_diff_days', 'acre_diff_abs', 'FINAL_ACRES']
incis_mtbs_c = (
    incis_mtbs
    .sort_values(sort_cols, ascending=[True, True, True, False])
    .drop_duplicates(subset=["MTBS_ID"], keep="first")
    .copy()
)
print(f"After resolving duplicates: {len(incis_mtbs_c)} unique MTBS_ID matches")

# --- Make sure matches are correct

# temporal filter
incis_mtbs_c = incis_mtbs_c[incis_mtbs_c['date_diff_days'] <= 14]

# acres filter (extreme cases)
q = incis_mtbs_c['acre_diff_pct'].quantile(0.70)
print(f"\n90th percentile absolute acre difference (%): {q}")
# incis_mtbs_c = incis_mtbs_c[incis_mtbs_c['acre_diff_abs'] < 20000] # less than 20,000 acre difference
incis_mtbs_c = incis_mtbs_c[
    (incis_mtbs_c['acre_diff_pct'] < q) |
    (incis_mtbs_c['acre_diff_abs'] < 10000)
] # less than 50% absolute difference or less than 10,000 acres
print(f"After removing big temporal/acre differences: {len(incis_mtbs_c)} unique MTBS_ID matches")
print(f"\n{incis_mtbs_c['acre_diff_abs'].describe()}\n")

# save this file out.
out_fp = os.path.join(projdir, 'data/spatial/mod/ics209plus-wf_incidents_2014to2023_draft022026_mtbs.gpkg')
os.makedirs(os.path.dirname(out_fp), exist_ok=True)
incis_mtbs_c.to_file(out_fp)
print(f"Saved to: {out_fp}")

MTBS up to 2024
Matched [4014/14303]
After resolving duplicates: 3958 unique MTBS_ID matches

90th percentile absolute acre difference (%): 11.799204725886675
After removing big temporal/acre differences: 3853 unique MTBS_ID matches

count    3,853.0000
mean       764.5589
std      1,652.3485
min          0.0000
25%         68.0000
50%        203.0000
75%        651.0000
max     27,945.0000
Name: acre_diff_abs, dtype: float64

Saved to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/spatial/mod/ics209plus-wf_incidents_2014to2023_draft022026_mtbs.gpkg


In [8]:
pd.set_option("display.float_format", "{:,.4f}".format)
print(incis_mtbs_c['acre_diff_abs'].describe())
print(incis_mtbs_c[incis_mtbs_c['INCIDENT_NAME'] == 'HERMITS PEAK'][['INCIDENT_NAME','BurnBndAc','FINAL_ACRES']].head())

count    3,853.0000
mean       764.5589
std      1,652.3485
min          0.0000
25%         68.0000
50%        203.0000
75%        651.0000
max     27,945.0000
Name: acre_diff_abs, dtype: float64
     INCIDENT_NAME  BurnBndAc  FINAL_ACRES
3596  HERMITS PEAK     352046 341,735.0000


In [9]:
# --- Filter to non-joined incident records
incis_ = incis.loc[
    ~incis["INCIDENT_ID"].isin(incis_mtbs_c["INCIDENT_ID"].unique())
].copy()
print(f"There are {len(incis_)} incidents remaining after MTBS join.")
# check if these have FOD or IRWIN information
print(f"Incidents with FOD linkage: {len(incis_[~incis_['FOD_ID'].isna()])}")
print(f"Incidents with IRWIN linkage: {len(incis_[~incis_['IRWIN_ID'].isna()])}")
print(f"Incidents with MTBS linkage: {len(incis_[~incis_['MTBS_ID'].isna()])}")
print(f"Incidents with neither: {len(incis_[(incis_['IRWIN_ID'].isna()) & (~incis_['FOD_ID'].isna())])}")

There are 10449 incidents remaining after MTBS join.
Incidents with FOD linkage: 7862
Incidents with IRWIN linkage: 9909
Incidents with MTBS linkage: 195
Incidents with neither: 480


### WUMI2024a

In [26]:
# --- Load the WUMI2024a fire perimeter data
fp = os.path.join(datadir, "wildfire/WUMI2024a/WUMI2024a_main_fires_unified_with_circles.gpkg")
wumi = gpd.read_file(fp, layer='WUMI2024a')
wumi = wumi[wumi['perimeter_type'] != 'CIRCLE'] # remove the circles
wumi['poly_area_ac'] = wumi['poly_area_ha'] * 2.47105 # make an acres column
print(wumi.columns)

# Subset columns
wumi = wumi[['fireid','dataset','name','date','poly_area_ac',
             'IRWINID','FOD_ID','MTBS_ID','fire_year','geometry']]

wumi['MTBS_ID'] = wumi['MTBS_ID'].str.upper()

# tidy date columns
wumi['date'] = pd.to_datetime(wumi['date'], format="mixed")
wumi['fire_year'] = wumi['fire_year'].astype(int)
wumi.rename(columns={'date':'WUMI_DATE','name':'WUMI_NAME', 'MTBS_ID':'WUMI_MTBS_ID'}, inplace=True)
wumi.head()

Index(['fireid', 'dataset', 'agency', 'name', 'date', 'lat', 'lon',
       'poly_area_ha', 'burn_area_ha', 'MTBS_name', 'MTBS_ID', 'IRWINID',
       'FOD_ID', 'FPA_ID', 'object_ID_wfaip', 'object_ID_fpafod',
       'object_ID_wfigs', 'object_ID_calfire', 'object_ID_usgs',
       'object_ID_iafph', 'cause_human_or_natural', 'cause_specific',
       'perimeter_type', 'fire_year', 'geometry', 'poly_area_ac'],
      dtype='object')


,fireid,dataset,WUMI_NAME,WUMI_DATE,poly_area_ac,IRWINID,FOD_ID,WUMI_MTBS_ID,fire_year,geometry
0,19840126_337090_1176180,mtbs,MODJESKA,1984-01-26,"1,266.9993",N/A,N/A,CA3370911761819840126,1984,"MULTIPOLYGON (((-1974416.021 1408348.28, -1974..."
2,19840127_342239_1181930,calfire,CREST,1984-01-27,605.7841,N/A,N/A,N/A,1984,"MULTIPOLYGON (((-2010684.866 1476422.231, -201..."
3,19840320_340151_1174086,calfire,RATTLESNAKE,1984-03-20,311.1200,N/A,N/A,N/A,1984,"MULTIPOLYGON (((-1946224.133 1436149.923, -194..."
6,19840329_360740_1201880,mtbs,FK 1817,1984-03-29,"1,461.9990",N/A,N/A,CA3607412018819840329,1984,"MULTIPOLYGON (((-2135612.942 1717960.609, -213..."
7,19840422_342699_1178510,calfire,UNKNOWN,1984-04-22,282.7763,N/A,N/A,N/A,1984,"MULTIPOLYGON (((-1979096.564 1472817.474, -197..."


In [28]:
print(len(wumi[wumi['WUMI_MTBS_ID'].isin(incis_mtbs_c["MTBS_ID"].unique())]))

2316


In [11]:
# --- Make sure incis_ is a real copy (avoids SettingWithCopyWarning)

incis_c = incis_.copy() # non-matched incidents
incis_c.drop(columns='geometry', inplace=True)

wumi = wumi.copy() # work on a copy

# --- Standardize the IRWIN and FOD IDs for joining
nulls = {"", "na", "n/a", "none", "null", "<null>",
         "(null)", "nan", "unknown", "unk"}

def clean_irwin(s: pd.Series) -> pd.Series:
    """
    Normalize IRWIN IDs to comparable GUID-like strings.
    Handles whitespace, case, braces, URL prefixes, and common fake-nulls.
    """
    out = (
        s.astype("string")
         .str.strip()
         .str.lower()
         .str.replace(r"\s+", "", regex=True)
         .str.replace("{", "", regex=False)
         .str.replace("}", "", regex=False)
         # If ever stored as a URL/path, keep only the last chunk
         .str.replace(r"^.*\/", "", regex=True)
    )
    return out.mask(out.isna() | out.isin(nulls), pd.NA)

def clean_fod(s: pd.Series) -> pd.Series:
    """Normalize FOD IDs to nullable integer."""
    return pd.to_numeric(s, errors="coerce").astype("Int64")

# --- Add cleaned keys to both WUMI and ICS
incis_c.loc[:, "IRWIN_C"] = clean_irwin(incis_c["IRWIN_ID"])
incis_c.loc[:, "FOD_C"] = clean_fod(incis_c["FOD_ID"])
wumi.loc[:, "IRWIN_C"] = clean_irwin(wumi["IRWINID"])
wumi.loc[:, "FOD_C"] = clean_fod(wumi["FOD_ID"])

# --- Dedupe WUMI perimeters on join keys (pick largest geometry for deterministic behavior)
# Use an equal-area CRS for area calculation
wumi_ae = wumi.to_crs("EPSG:5070").copy()
# select the best join if multiple
keep_cols = ['fireid','WUMI_DATE','WUMI_NAME','WUMI_MTBS_ID','poly_area_ac','geometry']
# by IRWIN
wumi_irwin_best = (
    wumi_ae[wumi_ae["IRWIN_C"].notna()]
    .sort_values(["IRWIN_C", "poly_area_ac"], ascending=[True, False])
    .drop_duplicates("IRWIN_C", keep="first")
    .loc[:, ["IRWIN_C"] + [c for c in keep_cols if c in wumi_ae.columns]]
    .copy()
)

# by FOD
wumi_fod_best = (
    wumi_ae[wumi_ae["FOD_C"].notna()]
    .sort_values(["FOD_C", "poly_area_ac"], ascending=[True, False])
    .drop_duplicates("FOD_C", keep="first")
    .loc[:, ["FOD_C"] + [c for c in keep_cols if c in wumi_ae.columns]]
    .copy()
)

# --- Join by IRWIN
inc_ir = incis_c.loc[incis_c["IRWIN_C"].notna()].copy()
j_ir = inc_ir.merge(
    wumi_irwin_best,
    on="IRWIN_C",
    how="inner",
    validate="m:1",
)
j_ir.loc[:, "match_method"] = "WUMI_IRWIN"
j_ir.loc[:, "match_key"] = j_ir["IRWIN_C"]
matched_ids = set(j_ir["INCIDENT_ID"].unique())

# --- Join remaining by FOD
inc_rem = incis_c.loc[~incis_c["INCIDENT_ID"].isin(matched_ids)].copy()
inc_fod = inc_rem.loc[inc_rem["FOD_C"].notna()].copy()
j_fod = inc_fod.merge(
    wumi_fod_best,
    on="FOD_C",
    how="inner",
    validate="m:1",
)
j_fod.loc[:, "match_method"] = "WUMI_FOD"
j_fod.loc[:, "match_key"] = j_fod["FOD_C"].astype("string")

# --- Combine + output as GeoDataFrame
incis_wumi = pd.concat([j_ir, j_fod], ignore_index=True)
print(incis_wumi.columns)
# IMPORTANT: geometry currently in EPSG:5070 because we merged from wumi_ae
incis_wumi = gpd.GeoDataFrame(incis_wumi, geometry="geometry", crs="EPSG:5070").to_crs("EPSG:4326")
print(f"Matched to WUMI via IRWIN: {j_ir['INCIDENT_ID'].nunique()}")
print(f"Matched to WUMI via FOD: {j_fod['INCIDENT_ID'].nunique()}")
print(f"Total matched to WUMI: {incis_wumi['INCIDENT_ID'].nunique()} / {len(incis_)}")

# --- Handle temporal/acre difference
incis_wumi["acre_diff_abs"] = (incis_wumi['FINAL_ACRES'] - incis_wumi["poly_area_ac"]).abs()
incis_wumi['date_diff_days'] = (incis_wumi["DISCOVERY_DATE"] - incis_wumi["WUMI_DATE"]).abs().dt.days
# symmetric percent diff (mean denominator) to avoid oddities
den = (incis_wumi["FINAL_ACRES"].abs() + incis_wumi["poly_area_ac"].abs()) / 2.0
incis_wumi["acre_diff_pct"] = 100.0 * incis_wumi["acre_diff_abs"] / den.replace(0, np.nan)

# --- Find good matches
# temporal filter
incis_wumi = incis_wumi[incis_wumi['date_diff_days'] <= 14]
# acres filter
q = incis_wumi['acre_diff_pct'].quantile(0.70)
print(f"\n90th percentile absolute acre difference (%): {q}")
# incis_mtbs_c = incis_mtbs_c[incis_mtbs_c['acre_diff_abs'] < 20000] # less than 20,000 acre difference
incis_wumi = incis_wumi[
    (incis_wumi['acre_diff_pct'] < q) |
    (incis_wumi['acre_diff_abs'] < 10000)
]# less than 50% absolute difference
print(f"After removing big temporal/acre differences: {len(incis_wumi)} unique WUMI matches")

# ---Remaining incidents to carry forward
incis_rm = incis_.loc[~incis_["INCIDENT_ID"].isin(incis_wumi["INCIDENT_ID"].unique())].copy()
print(f"Remaining after WUMI join: {len(incis_rm)}")

# save the geopackage out.
out_fp = os.path.join(projdir, 'data/spatial/mod/ics209plus-wf_incidents_2014to2023_draft022026_wumi.gpkg')
incis_wumi.to_file(out_fp)
print(f"Saved to {out_fp}")

Index(['CAUSE', 'COMPLEX', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
       'EVACUATION_REPORTED', 'EXPECTED_CONTAINMENT_DATE', 'FATALITIES',
       'FATALITIES_PUBLIC', 'FATALITIES_RESPONDER', 'FINAL_ACRES',
       'FINAL_REPORT_DATE', 'FIRE_SIZE_CLASS', 'FIRE_YEAR', 'FOD_CONT_DATE',
       'FOD_CONT_DOY', 'FOD_CONT_TIME', 'FOD_DISCOVERY_DATE',
       'FOD_DISCOVERY_DOY', 'FOD_DISCOVERY_TIME', 'FOD_EXCL_CAT',
       'FOD_FIRE_SIZE', 'FOD_ID', 'FOD_INCIDENT_GROUP_NAME', 'FOD_LATITUDE',
       'FOD_LONGITUDE', 'FUEL_MODEL', 'INCIDENT_DESCRIPTION', 'INCIDENT_ID',
       'INCIDENT_ID_OLD', 'INCIDENT_NAME', 'INCIDENT_NUMBER',
       'INCTYP_ABBREVIATION', 'INC_IDENTIFIER', 'INC_MGMT_NUM_SITREPS',
       'INJURIES_TOTAL', 'IRWIN_ID', 'LL_CONFIDENCE', 'LL_UPDATE',
       'LOCAL_TIMEZONE', 'MTBS_FIRE_NAME', 'MTBS_ID',
       'NWCG_CAUSE_CLASSIFICATION', 'NWCG_GENERAL_CAUSE', 'PEAK_EVACUATIONS',
       'POO_CITY', 'POO_COUNTY', 'POO_LATITUDE', 'POO_LONGITUDE',
       'POO_SHORT_LOCATION_DESC', 'POO_S

### NIFC Perimeters

In [12]:
# --- Bring in the NIFC Interagency Fire Perimeter History
fp = os.path.join(datadir, 'wildfire/NIFC/InterAgencyFirePerimeterHistory_All.gpkg')
nifc = gpd.read_file(fp)
print(nifc.columns)

# --- Temporal filter
incis_rm['START_YEAR'] = incis_rm['START_YEAR'].astype(int)
nifc['FIRE_YEAR_INT'] = nifc['FIRE_YEAR_INT'].fillna(0)
nifc['FIRE_YEAR_INT'] = nifc['FIRE_YEAR_INT'].astype(int)
nifc = nifc[(nifc['FIRE_YEAR_INT'] >= incis_rm['START_YEAR'].min()) &
            (nifc['FIRE_YEAR_INT'] <= incis_rm['START_YEAR'].max())]

# --- Retain records with an IRWIN ID
print(f"{int(nifc['IRWINID'].isna().sum())}/{len(nifc)} fires with IRWINID")
nifc_irwin = nifc[~nifc['IRWINID'].isna()]

# --- Format the IRWIN ID
# ICS incidents
incis_rm_ = incis_rm.copy()
incis_rm_.drop(columns='geometry', inplace=True)
incis_rm_.loc[:, "IRWIN_C"] = clean_irwin(incis_rm_["IRWIN_ID"])
# NIFC perimeters
nifc_irwin = nifc_irwin.copy()
nifc_irwin.loc[:, "IRWIN_C"] = clean_irwin(nifc_irwin["IRWINID"])

# --- Keep best match for duplicate IRWIN IDs
nifc_ae = nifc_irwin.to_crs("EPSG:5070").copy()
nifc_ae["area_m2"] = nifc_ae.geometry.area

nifc_irwin = (
    nifc_ae[nifc_ae["IRWIN_C"].notna()]
    .sort_values(["IRWIN_C", "area_m2"], ascending=[True, False])
    .drop_duplicates("IRWIN_C", keep="first")
    .copy()
)

# --- Join to the incident table
incis_nifc = incis_rm_.merge(
    nifc_irwin,
    on='IRWIN_C',
    how="inner",
    validate='m:1'
)
incis_nifc = gpd.GeoDataFrame(incis_nifc, geometry="geometry", crs="EPSG:5070").to_crs("EPSG:4326")
print(f"Found {len(incis_nifc)} matching IRWINID")

# --- Handle temporal/acre difference
incis_nifc["acre_diff_abs"] = (incis_nifc['FINAL_ACRES'] - incis_nifc["GIS_ACRES"]).abs()
# symmetric percent diff (mean denominator) to avoid oddities
den = (incis_nifc["FINAL_ACRES"].abs() + incis_nifc["GIS_ACRES"].abs()) / 2.0
incis_nifc["acre_diff_pct"] = 100.0 * incis_nifc["acre_diff_abs"] / den.replace(0, np.nan)


# --- Find good matches
# temporal filter (by year for NIFC)
incis_nifc = incis_nifc[incis_nifc['START_YEAR'] == incis_nifc['FIRE_YEAR_INT']]
# acres filter
q = incis_nifc['acre_diff_pct'].quantile(0.70)
print(f"\n97th percentile absolute acre difference (%): {q}")
# incis_mtbs_c = incis_mtbs_c[incis_mtbs_c['acre_diff_abs'] < 20000] # less than 20,000 acre difference
incis_nifc = incis_nifc[
    (incis_nifc['acre_diff_pct'] < q) |
    (incis_nifc['acre_diff_abs'] < 10000)
] # less than 50% absolute difference
print(f"After removing big temporal/acre differences: {len(incis_nifc)} unique NIFC matches")

# ---Remaining incidents to carry forward
incis_rm_ = incis_rm.loc[~incis_rm["INCIDENT_ID"].isin(incis_nifc["INCIDENT_ID"].unique())].copy()
print(f"Remaining after NIFC join: {len(incis_rm_)}")
print(len(incis)-len(incis_rm_))

# save this file out.
out_fp = os.path.join(projdir, 'data/spatial/mod/ics209plus-wf_incidents_2014to2023_draft022026_nifc.gpkg')
incis_nifc.to_file(out_fp)
print(f"Saved to {out_fp}")

Index(['IRWINID', 'FORID', 'INCIDENT', 'GIS_ACRES', 'UNQE_FIRE_ID', 'DATE_CUR',
       'FIRE_YEAR_INT', 'UNIT_ID', 'POO_RESP_I', 'LOCAL_NUM', 'FEATURE_CA',
       'MAP_METHOD', 'COMMENTS', 'GEO_ID', 'SOURCE', 'AGENCY', 'FIRE_YEAR',
       'GlobalID', 'geometry'],
      dtype='object')
11142/30760 fires with IRWINID
Found 2152 matching IRWINID

97th percentile absolute acre difference (%): 6.0899240544787725
After removing big temporal/acre differences: 2113 unique NIFC matches
Remaining after NIFC join: 7362
6941
Saved to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/spatial/mod/ics209plus-wf_incidents_2014to2023_draft022026_nifc.gpkg


In [13]:
pd.set_option("display.float_format", "{:,.4f}".format)
print(incis_nifc['acre_diff_abs'].describe())
print(incis_wumi['acre_diff_abs'].describe())
print(incis_mtbs_c['acre_diff_abs'].describe())

count   2,113.0000
mean      126.3824
std       734.1919
min         0.0000
25%         0.2500
50%         1.3300
75%        16.4100
max     9,929.0000
Name: acre_diff_abs, dtype: float64
count      974.0000
mean       189.2752
std      1,331.2369
min          0.0013
25%          0.4541
50%          5.6065
75%         64.7572
max     36,144.2949
Name: acre_diff_abs, dtype: float64
count    3,853.0000
mean       764.5589
std      1,652.3485
min          0.0000
25%         68.0000
50%        203.0000
75%        651.0000
max     27,945.0000
Name: acre_diff_abs, dtype: float64


In [14]:
# check remaining fires
incis_rm_ = incis_rm[~incis_rm['INCIDENT_ID'].isin(incis_nifc["INCIDENT_ID"].unique())]
print(f"There are {len(incis_rm_)} incidents remaining")
print(len(incis)-len(incis_rm_))

There are 7362 incidents remaining
6941


In [15]:
# --- Merge each

# make sure we have the accurate match methods
incis_mtbs_c['match_method'] = 'MTBS'
incis_nifc['match_method'] = 'NIFC_IRWIN'
incis_wumi['match_method'] = incis_wumi['match_method']

# make sure crs aligns
incis_mtbs_c = incis_mtbs_c.to_crs(proj_crs)
incis_nifc = incis_nifc.to_crs(proj_crs)
incis_wumi = incis_wumi.to_crs(proj_crs)

# concatenate each of them
incis_join = pd.concat([incis_mtbs_c, incis_nifc, incis_wumi], ignore_index=True)

# tidy columns
ics_columns = list(incis.columns)
incis_join = incis_join[
    ics_columns + [
        "match_method",
        "acre_diff_abs",
        "acre_diff_pct",
        "date_diff_days",
        "match_key",
        "IRWIN_C",
        "FOD_C",
        "WUMI_MTBS_ID"
    ]
]

print(f"Final joined rows (before LC filtering): {len(incis_join)}")

Final joined rows (before LC filtering): 6940


In [16]:
for c in incis_join.columns:
    print(c)

CAUSE
COMPLEX
DISCOVERY_DATE
DISCOVERY_DOY
EVACUATION_REPORTED
EXPECTED_CONTAINMENT_DATE
FATALITIES
FATALITIES_PUBLIC
FATALITIES_RESPONDER
FINAL_ACRES
FINAL_REPORT_DATE
FIRE_SIZE_CLASS
FIRE_YEAR
FOD_CONT_DATE
FOD_CONT_DOY
FOD_CONT_TIME
FOD_DISCOVERY_DATE
FOD_DISCOVERY_DOY
FOD_DISCOVERY_TIME
FOD_EXCL_CAT
FOD_FIRE_SIZE
FOD_ID
FOD_INCIDENT_GROUP_NAME
FOD_LATITUDE
FOD_LONGITUDE
FUEL_MODEL
INCIDENT_DESCRIPTION
INCIDENT_ID
INCIDENT_ID_OLD
INCIDENT_NAME
INCIDENT_NUMBER
INCTYP_ABBREVIATION
INC_IDENTIFIER
INC_MGMT_NUM_SITREPS
INJURIES_TOTAL
IRWIN_ID
LL_CONFIDENCE
LL_UPDATE
LOCAL_TIMEZONE
MTBS_FIRE_NAME
MTBS_ID
NWCG_CAUSE_CLASSIFICATION
NWCG_GENERAL_CAUSE
PEAK_EVACUATIONS
POO_CITY
POO_COUNTY
POO_LATITUDE
POO_LONGITUDE
POO_SHORT_LOCATION_DESC
POO_STATE
PROJECTED_FINAL_IM_COST
START_YEAR
STR_DAMAGED_COMM_TOTAL
STR_DAMAGED_RES_TOTAL
STR_DAMAGED_TOTAL
STR_DESTROYED_COMM_TOTAL
STR_DESTROYED_RES_TOTAL
STR_DESTROYED_TOTAL
STR_THREATENED_COMM_MAX
STR_THREATENED_MAX
STR_THREATENED_RES_MAX
SUPPRESSION_MET

In [17]:
# --- Check on acre diff
pd.set_option("display.float_format", "{:,.4f}".format)
print(incis_join['acre_diff_abs'].describe())

count    6,940.0000
mean       489.5167
std      1,422.3244
min          0.0000
25%          3.0000
50%         63.0000
75%        304.0000
max     36,144.2949
Name: acre_diff_abs, dtype: float64


In [18]:
# --- Check for duplicate MTBS IDs
dup_counts = incis_join["INCIDENT_ID"].value_counts()
dup_incis = dup_counts[dup_counts > 1]
print(f"\tDuplicated INCIDENT IDs: {len(dup_incis)}")
print(f"\tICS incidents involved: {dup_incis.sum()}")

	Duplicated INCIDENT IDs: 0
	ICS incidents involved: 0


In [19]:
dups = pd.DataFrame(dup_incis).reset_index()
incis_dup = incis_join[incis_join['INCIDENT_ID'].isin(dups['INCIDENT_ID'].unique())]
print(incis_dup[['INCIDENT_ID','match_method','FINAL_ACRES']].head(50))

Empty DataFrame
Columns: [INCIDENT_ID, match_method, FINAL_ACRES]
Index: []


In [20]:
# --- save out
out_fp = os.path.join(projdir, 'data/spatial/mod/ics209plus_2014to2023_perimeters.gpkg')
incis_join.to_file(out_fp)
print(f"Saved to {out_fp}")

Saved to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/spatial/mod/ics209plus_2014to2023_perimeters.gpkg


In [21]:
incis_join['match_method'].value_counts()

match_method
MTBS          3853
NIFC_IRWIN    2113
WUMI_IRWIN     770
WUMI_FOD       204
Name: count, dtype: int64

In [22]:
# --- Export a clean shapefile for GEE
incis_join_ee = incis_join[['INCIDENT_ID','START_YEAR','geometry']].copy()
incis_join_ee.rename(columns={'INCIDENT_ID':'inci_id'}, inplace=True)

out_fp = os.path.join(projdir, 'data/spatial/mod/ee/incidents_joined_EE.shp')
os.makedirs(os.path.dirname(out_fp), exist_ok=True)
incis_join_ee.to_file(out_fp)
print(f"Saved to {out_fp}")

Saved to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/spatial/mod/ee/incidents_joined_EE.shp
